In [1]:
"""
FinRL Tutorial - Part 2: Model Training (수정본)
Stock NeurIPS2018 시리즈 - DRL 모델 학습

이 파일은 5개의 DRL 알고리즘(A2C, DDPG, PPO, TD3, SAC)을 사용하여 트레이딩 에이전트를 학습합니다.
"""

'\nFinRL Tutorial - Part 2: Model Training (수정본)\nStock NeurIPS2018 시리즈 - DRL 모델 학습\n\n이 파일은 5개의 DRL 알고리즘(A2C, DDPG, PPO, TD3, SAC)을 사용하여 트레이딩 에이전트를 학습합니다.\n'

# Stock NeurIPS2018 Part 2. Train (수정본)

이 노트북은 FinRL을 사용하여 주식 트레이딩을 위한 DRL 에이전트를 학습합니다.

**주요 수정 사항:**
- gymnasium 호환성 개선
- stable-baselines3 최신 API 적용
- 학습 진행 상황 로깅 추가
- 모델 저장 검증 추가

## Part 0. 실행 전 설정

In [2]:
import json
import os
from datetime import datetime
from pathlib import Path

# 진행 상황 추적
PROGRESS_LOG_FILE = "PROGRESS_LOG.md"

def log_progress(phase, step, status, message=""):
    """진행 상황을 마크다운 파일에 기록"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"\n### [{timestamp}] Phase {phase}, Step {step}\n"
    log_entry += f"**Status**: {status}\n"
    if message:
        log_entry += f"**Message**: {message}\n"

    with open(PROGRESS_LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(log_entry)

    print(f"[{status}] Phase {phase}.{step}: {message}")

log_progress(2, 0, "START", "DRL 모델 학습 시작")

[START] Phase 2.0: DRL 모델 학습 시작


## Part 1. 패키지 설치 및 Import

In [3]:
import sys

log_progress(2, 1, "RUNNING", "필수 패키지 설치 중...")

# 필수 패키지 설치
packages = [
    "stable-baselines3>=2.0.0",
    "gymnasium>=0.28.1",
    "shimmy>=1.3.0",
]

for package in packages:
    print(f"Installing {package}...")
    !{sys.executable} -m pip install {package} -q

# FinRL 설치 (아직 설치되지 않은 경우)
!{sys.executable} -m pip install git+https://github.com/AI4Finance-Foundation/FinRL.git -q

log_progress(2, 1, "COMPLETED", "패키지 설치 완료")

[RUNNING] Phase 2.1: 필수 패키지 설치 중...
Installing stable-baselines3>=2.0.0...
Installing gymnasium>=0.28.1...
Installing shimmy>=1.3.0...
[COMPLETED] Phase 2.1: 패키지 설치 완료


In [4]:
# Import
log_progress(2, 2, "RUNNING", "패키지 import 중...")

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

try:
    from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
    from finrl.agents.stablebaselines3.models import DRLAgent
    from stable_baselines3.common.logger import configure
    from finrl import config_tickers
    from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR
    from finrl.main import check_and_make_directories

    # stable-baselines3 모델들
    from stable_baselines3 import A2C, DDPG, PPO, SAC, TD3

    print("모든 패키지 import 성공!")
    log_progress(2, 2, "SUCCESS", "패키지 import 완료")

except ImportError as e:
    error_msg = f"Import 실패: {str(e)}"
    print(error_msg)
    log_progress(2, 2, "ERROR", error_msg)
    raise

[RUNNING] Phase 2.2: 패키지 import 중...
모든 패키지 import 성공!
[SUCCESS] Phase 2.2: 패키지 import 완료


In [5]:
# GPU 확인 및 모니터링 설정
log_progress(2, 2.5, "RUNNING", "GPU 확인 중...")

import torch

# GPU 사용 가능 여부 확인
if not torch.cuda.is_available():
    error_msg = "❌ GPU를 사용할 수 없습니다! CUDA가 설치되어 있는지 확인하세요."
    print(error_msg)
    log_progress(2, 2.5, "ERROR", error_msg)
    raise RuntimeError(error_msg)

# GPU 정보 출력
device = torch.device('cuda')
print(f"\n{'='*60}")
print(f"✅ GPU 사용 가능!")
print(f"{'='*60}")
print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
print(f"GPU 개수: {torch.cuda.device_count()}")
print(f"현재 GPU: cuda:{torch.cuda.current_device()}")
print(f"총 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"{'='*60}\n")

# GPU 메모리 모니터링 함수
def print_gpu_memory(stage=""):
    """GPU 메모리 사용량 출력"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"\n[GPU 메모리 - {stage}]")
        print(f"  할당됨: {allocated:.2f} GB")
        print(f"  예약됨: {reserved:.2f} GB")
        print(f"  총 용량: {total:.2f} GB")
        print(f"  사용률: {(reserved/total)*100:.1f}%\n")

print("GPU 메모리 모니터링 함수 정의 완료")
print_gpu_memory("초기 상태")

log_progress(2, 2.5, "SUCCESS", f"GPU 확인 완료: {torch.cuda.get_device_name(0)}")

[RUNNING] Phase 2.2.5: GPU 확인 중...

✅ GPU 사용 가능!
GPU 이름: NVIDIA H100 80GB HBM3
GPU 개수: 1
현재 GPU: cuda:0
총 메모리: 79.18 GB

GPU 메모리 모니터링 함수 정의 완료

[GPU 메모리 - 초기 상태]
  할당됨: 0.00 GB
  예약됨: 0.00 GB
  총 용량: 79.18 GB
  사용률: 0.0%

[SUCCESS] Phase 2.2.5: GPU 확인 완료: NVIDIA H100 80GB HBM3


In [6]:
# Custom Callback for Off-Policy Algorithms
log_progress(2, 2.6, "RUNNING", "Custom Callback 클래스 정의 중...")

from stable_baselines3.common.callbacks import BaseCallback
import statistics

class OffPolicyTensorboardCallback(BaseCallback):
    """
    Off-policy 알고리즘(DDPG, TD3, SAC)과 on-policy 알고리즘(A2C, PPO) 모두 지원하는 Custom callback.
    
    FinRL의 기본 TensorboardCallback은 rollout_buffer만 지원하여
    off-policy 알고리즘에서 'Logging Error: rollout_buffer' 에러가 발생하는 문제를 해결합니다.
    """
    
    def __init__(self, verbose=0):
        super().__init__(verbose)
        
    def _on_step(self) -> bool:
        """매 스텝마다 호출 - reward 로깅"""
        try:
            self.logger.record(key="train/reward", value=self.locals["rewards"][0])
        except BaseException as error:
            try:
                self.logger.record(key="train/reward", value=self.locals["reward"][0])
            except BaseException:
                # reward를 찾을 수 없는 경우 무시
                pass
        return True
    
    def _on_rollout_end(self) -> bool:
        """
        Rollout 종료 시 호출 - on-policy 알고리즘에서만 작동
        Off-policy 알고리즘은 rollout_buffer가 없으므로 조용히 스킵
        """
        try:
            # On-policy 알고리즘 (A2C, PPO)은 rollout_buffer 사용
            if "rollout_buffer" in self.locals:
                rollout_buffer_rewards = self.locals["rollout_buffer"].rewards.flatten()
                self.logger.record(
                    key="train/reward_min", value=min(rollout_buffer_rewards)
                )
                self.logger.record(
                    key="train/reward_mean", value=statistics.mean(rollout_buffer_rewards)
                )
                self.logger.record(
                    key="train/reward_max", value=max(rollout_buffer_rewards)
                )
            # Off-policy 알고리즘 (DDPG, TD3, SAC)은 replay_buffer 사용
            # replay_buffer에서는 reward 통계를 다르게 수집해야 하므로 스킵
        except Exception:
            # 에러 발생 시 조용히 무시 (에러 메시지 출력 안 함)
            pass
        return True

print("✅ OffPolicyTensorboardCallback 클래스 정의 완료")
print("   - On-policy 알고리즘 (A2C, PPO): rollout_buffer 사용")
print("   - Off-policy 알고리즘 (DDPG, TD3, SAC): 에러 없이 스킵")
log_progress(2, 2.6, "SUCCESS", "Custom Callback 클래스 정의 완료")

[RUNNING] Phase 2.2.6: Custom Callback 클래스 정의 중...
✅ OffPolicyTensorboardCallback 클래스 정의 완료
   - On-policy 알고리즘 (A2C, PPO): rollout_buffer 사용
   - Off-policy 알고리즘 (DDPG, TD3, SAC): 에러 없이 스킵
[SUCCESS] Phase 2.2.6: Custom Callback 클래스 정의 완료


In [7]:
# 디렉토리 생성
check_and_make_directories([TRAINED_MODEL_DIR, RESULTS_DIR])
print(f"모델 저장 디렉토리: {TRAINED_MODEL_DIR}")
print(f"결과 저장 디렉토리: {RESULTS_DIR}")

모델 저장 디렉토리: trained_models
결과 저장 디렉토리: results


## Part 2. OpenAI Gym 스타일 환경 구축

주식 트레이딩 환경을 구축합니다.

### 2.1 학습 데이터 로드

In [8]:
log_progress(2, 3, "RUNNING", "학습 데이터 로드 중...")

try:
    train = pd.read_csv('train_data.csv')
    print(f"학습 데이터 로드 완료: {len(train)}행")
    print(f"기간: {train['date'].min()} ~ {train['date'].max()}")

    # FinRL 환경을 위한 인덱스 생성
    # 각 날짜에 순차적인 day 번호를 할당하여 인덱스로 설정
    train = train.sort_values(['date', 'tic']).reset_index(drop=True)
    date_to_day = {date: i for i, date in enumerate(sorted(train['date'].unique()))}
    train['day'] = train['date'].map(date_to_day)
    train = train.set_index('day')
    
    print(f"인덱스 생성 완료: {train.index.nunique()}개 거래일")

    log_progress(2, 3, "SUCCESS", f"데이터 로드 완료 ({len(train)}행)")

except FileNotFoundError:
    error_msg = "train_data.csv 파일을 찾을 수 없습니다. Part 1을 먼저 실행하세요."
    print(error_msg)
    log_progress(2, 3, "ERROR", error_msg)
    raise

except Exception as e:
    error_msg = f"데이터 로드 실패: {str(e)}"
    print(error_msg)
    log_progress(2, 3, "ERROR", error_msg)
    raise

[RUNNING] Phase 2.3: 학습 데이터 로드 중...
학습 데이터 로드 완료: 81004행
기간: 2009-01-02 ~ 2020-06-30
인덱스 생성 완료: 2893개 거래일
[SUCCESS] Phase 2.3: 데이터 로드 완료 (81004행)


### 2.2 환경 파라미터 설정

In [9]:
log_progress(2, 4, "RUNNING", "환경 파라미터 계산 중...")

# 주식 개수 및 상태 공간 계산
stock_dimension = len(train.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension

print(f"\n=== 환경 파라미터 ===")
print(f"주식 종목 수 (Stock Dimension): {stock_dimension}")
print(f"상태 공간 크기 (State Space): {state_space}")
print(f"기술적 지표 수: {len(INDICATORS)}")
print(f"지표 목록: {INDICATORS}")

# 거래 비용 및 제약 조건
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,                          # 한 번에 거래 가능한 최대 주식 수
    "initial_amount": 1000000,             # 초기 자본금
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,        # 매수 수수료 (0.1%)
    "sell_cost_pct": sell_cost_list,      # 매도 수수료 (0.1%)
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4                 # 보상 스케일링
}

print(f"\n초기 자본금: ${env_kwargs['initial_amount']:,}")
print(f"거래 수수료: {env_kwargs['buy_cost_pct'][0]*100}%")
print(f"최대 거래 수량: {env_kwargs['hmax']}주")

log_progress(2, 4, "SUCCESS", f"환경 파라미터 설정 완료 (state_space: {state_space})")

[RUNNING] Phase 2.4: 환경 파라미터 계산 중...

=== 환경 파라미터 ===
주식 종목 수 (Stock Dimension): 28
상태 공간 크기 (State Space): 281
기술적 지표 수: 8
지표 목록: ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']

초기 자본금: $1,000,000
거래 수수료: 0.1%
최대 거래 수량: 100주
[SUCCESS] Phase 2.4: 환경 파라미터 설정 완료 (state_space: 281)


In [10]:
# 학습 환경 생성
log_progress(2, 5, "RUNNING", "학습 환경 생성 중...")

try:
    e_train_gym = StockTradingEnv(df=train, **env_kwargs)
    env_train, _ = e_train_gym.get_sb_env()

    print(f"환경 타입: {type(env_train)}")
    print("학습 환경 생성 완료!")

    log_progress(2, 5, "SUCCESS", "학습 환경 생성 완료")

except Exception as e:
    error_msg = f"환경 생성 실패: {str(e)}"
    print(error_msg)
    log_progress(2, 5, "ERROR", error_msg)
    raise

[RUNNING] Phase 2.5: 학습 환경 생성 중...
환경 타입: <class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>
학습 환경 생성 완료!
[SUCCESS] Phase 2.5: 학습 환경 생성 완료


## Part 3. DRL 에이전트 학습

5개의 알고리즘으로 에이전트를 학습합니다.

In [11]:
# 에이전트 초기화
agent = DRLAgent(env=env_train)

# 학습할 알고리즘 선택
if_using_a2c = True
if_using_ddpg = True
if_using_ppo = True
if_using_td3 = True
if_using_sac = True

print("\n=== 학습할 알고리즘 ===")
algorithms = []
if if_using_a2c: algorithms.append("A2C")
if if_using_ddpg: algorithms.append("DDPG")
if if_using_ppo: algorithms.append("PPO")
if if_using_td3: algorithms.append("TD3")
if if_using_sac: algorithms.append("SAC")
print(f"선택된 알고리즘: {', '.join(algorithms)}")


=== 학습할 알고리즘 ===
선택된 알고리즘: A2C, DDPG, PPO, TD3, SAC


### 3.1 A2C (Advantage Actor-Critic)

In [11]:
if if_using_a2c:
    log_progress(3, 1, "RUNNING", "A2C 모델 학습 시작...")

    try:
        # GPU 메모리 확인
        print_gpu_memory("A2C 학습 시작 전")
        
        # 모델 생성 (GPU 명시)
        agent = DRLAgent(env=env_train)
        model_a2c = agent.get_model("a2c", model_kwargs={'device': 'cuda'})

        # 로거 설정
        tmp_path = RESULTS_DIR + '/a2c'
        new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        model_a2c.set_logger(new_logger_a2c)

        print("A2C 모델 파라미터:", model_a2c.get_parameters())
        print(f"✅ A2C 모델이 GPU에서 학습됩니다: {model_a2c.device}")

        # 학습 실행 (50,000 timesteps)
        trained_a2c = agent.train_model(
            model=model_a2c,
            tb_log_name='a2c',
            total_timesteps=50000
        )

        # 모델 저장
        save_path = TRAINED_MODEL_DIR + "/agent_a2c"
        trained_a2c.save(save_path)

        # 저장 확인
        if os.path.exists(save_path + ".zip"):
            file_size = os.path.getsize(save_path + ".zip") / 1024  # KB
            print(f"\nA2C 모델 저장 완료: {save_path}.zip ({file_size:.2f} KB)")
            
            # GPU 메모리 확인
            print_gpu_memory("A2C 학습 완료")
            
            log_progress(3, 1, "SUCCESS", f"A2C 학습 완료 및 저장 ({file_size:.2f}KB)")
        else:
            raise FileNotFoundError("모델 파일이 저장되지 않았습니다")

    except Exception as e:
        error_msg = f"A2C 학습 실패: {str(e)}"
        print(error_msg)
        log_progress(3, 1, "ERROR", error_msg)
else:
    print("A2C 학습 건너뜀")

[RUNNING] Phase 3.1: A2C 모델 학습 시작...

[GPU 메모리 - A2C 학습 시작 전]
  할당됨: 0.00 GB
  예약됨: 0.00 GB
  총 용량: 79.18 GB
  사용률: 0.0%

{'device': 'cuda'}
Using cuda device
Logging to results/a2c
A2C 모델 파라미터: {'policy': OrderedDict([('log_std', tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], device='cuda:0')), ('mlp_extractor.policy_net.0.weight', tensor([[ 0.0734, -0.0004,  0.0866,  ..., -0.0641, -0.1342,  0.0817],
        [ 0.0366,  0.0702,  0.1999,  ..., -0.0382, -0.0742,  0.0756],
        [-0.0704, -0.0721,  0.0761,  ..., -0.1032, -0.0351,  0.2037],
        ...,
        [-0.0568, -0.0463, -0.0193,  ..., -0.2520,  0.0979,  0.1451],
        [ 0.0422,  0.0142,  0.0682,  ...,  0.0211, -0.0012,  0.1608],
        [ 0.0330, -0.0182, -0.0662,  ...,  0.1426, -0.0523, -0.0343]],
       device='cuda:0')), ('mlp_extractor.policy_net.0.bias', tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

### 3.2 DDPG (Deep Deterministic Policy Gradient)

In [12]:
if if_using_ddpg:
    log_progress(3, 2, "RUNNING", "DDPG 모델 학습 시작...")

    try:
        # GPU 메모리 확인
        print_gpu_memory("DDPG 학습 시작 전")
        
        # 모델 생성 (GPU 명시)
        agent = DRLAgent(env=env_train)
        model_ddpg = agent.get_model("ddpg", model_kwargs={'device': 'cuda'})

        # 로거 설정 (off-policy 알고리즘은 replay buffer를 사용하므로 CSV 로거 제외)
        tmp_path = RESULTS_DIR + '/ddpg'
        new_logger_ddpg = configure(tmp_path, ["stdout", "tensorboard"])
        model_ddpg.set_logger(new_logger_ddpg)
        
        print(f"✅ DDPG 모델이 GPU에서 학습됩니다: {model_ddpg.device}")
        print("📝 Custom Callback 사용 (rollout_buffer 에러 방지)")

        # 학습 실행 - Custom Callback 사용
        trained_ddpg = model_ddpg.learn(
            total_timesteps=50000,
            tb_log_name='ddpg',
            callback=OffPolicyTensorboardCallback()
        )

        # 모델 저장
        save_path = TRAINED_MODEL_DIR + "/agent_ddpg"
        trained_ddpg.save(save_path)

        if os.path.exists(save_path + ".zip"):
            file_size = os.path.getsize(save_path + ".zip") / 1024
            print(f"\n✅ DDPG 모델 저장 완료: {save_path}.zip ({file_size:.2f} KB)")
            
            # GPU 메모리 확인
            print_gpu_memory("DDPG 학습 완료")
            
            log_progress(3, 2, "SUCCESS", f"DDPG 학습 완료 및 저장 ({file_size:.2f}KB)")
        else:
            raise FileNotFoundError("모델 파일이 저장되지 않았습니다")

    except Exception as e:
        error_msg = f"DDPG 학습 실패: {str(e)}"
        print(error_msg)
        log_progress(3, 2, "ERROR", error_msg)
else:
    print("DDPG 학습 건너뜀")

[RUNNING] Phase 3.2: DDPG 모델 학습 시작...

[GPU 메모리 - DDPG 학습 시작 전]
  할당됨: 0.00 GB
  예약됨: 0.00 GB
  총 용량: 79.18 GB
  사용률: 0.0%

{'device': 'cuda'}
Using cuda device
Logging to results/ddpg
✅ DDPG 모델이 GPU에서 학습됩니다: cuda
📝 Custom Callback 사용 (rollout_buffer 에러 방지)
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 92        |
|    time_elapsed    | 125       |
|    total_timesteps | 11572     |
| train/             |           |
|    actor_loss      | 557       |
|    critic_loss     | 337       |
|    learning_rate   | 0.001     |
|    n_updates       | 11471     |
|    reward          | 7.6384006 |
----------------------------------
----------------------------------
| time/              |           |
|    episodes        | 8         |
|    fps             | 92        |
|    time_elapsed    | 251       |
|    total_timesteps | 23144     |
| train/             |           |
|    actor_loss      | 326       |
|    cr

### 3.3 PPO (Proximal Policy Optimization)

In [13]:
if if_using_ppo:
    log_progress(3, 3, "RUNNING", "PPO 모델 학습 시작...")

    try:
        # GPU 메모리 확인
        print_gpu_memory("PPO 학습 시작 전")
        
        agent = DRLAgent(env=env_train)

        # PPO 하이퍼파라미터 (GPU 명시)
        PPO_PARAMS = {
            "n_steps": 2048,
            "ent_coef": 0.01,
            "learning_rate": 0.00025,
            "batch_size": 128,
            "device": "cuda"
        }

        model_ppo = agent.get_model("ppo", model_kwargs=PPO_PARAMS)

        # 로거 설정
        tmp_path = RESULTS_DIR + '/ppo'
        new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        model_ppo.set_logger(new_logger_ppo)
        
        print(f"✅ PPO 모델이 GPU에서 학습됩니다: {model_ppo.device}")

        # 학습 실행 (200,000 timesteps - 더 오래 학습)
        print("PPO는 다른 알고리즘보다 학습 시간이 더 걸립니다...")
        trained_ppo = agent.train_model(
            model=model_ppo,
            tb_log_name='ppo',
            total_timesteps=200000
        )

        # 모델 저장
        save_path = TRAINED_MODEL_DIR + "/agent_ppo"
        trained_ppo.save(save_path)

        if os.path.exists(save_path + ".zip"):
            file_size = os.path.getsize(save_path + ".zip") / 1024
            print(f"\nPPO 모델 저장 완료: {save_path}.zip ({file_size:.2f} KB)")
            
            # GPU 메모리 확인
            print_gpu_memory("PPO 학습 완료")
            
            log_progress(3, 3, "SUCCESS", f"PPO 학습 완료 및 저장 ({file_size:.2f}KB)")
        else:
            raise FileNotFoundError("모델 파일이 저장되지 않았습니다")

    except Exception as e:
        error_msg = f"PPO 학습 실패: {str(e)}"
        print(error_msg)
        log_progress(3, 3, "ERROR", error_msg)
else:
    print("PPO 학습 건너뜀")

[RUNNING] Phase 3.3: PPO 모델 학습 시작...

[GPU 메모리 - PPO 학습 시작 전]
  할당됨: 0.07 GB
  예약됨: 0.08 GB
  총 용량: 79.18 GB
  사용률: 0.1%

{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 128, 'device': 'cuda'}
Using cuda device
Logging to results/ppo
✅ PPO 모델이 GPU에서 학습됩니다: cuda
PPO는 다른 알고리즘보다 학습 시간이 더 걸립니다...
------------------------------------
| time/              |             |
|    fps             | 158         |
|    iterations      | 1           |
|    time_elapsed    | 12          |
|    total_timesteps | 2048        |
| train/             |             |
|    reward          | -0.23598708 |
|    reward_max      | 8.705552    |
|    reward_mean     | 0.0958951   |
|    reward_min      | -9.001229   |
------------------------------------
day: 2892, episode: 20
begin_total_asset: 1000000.00
end_total_asset: 3997903.57
total_reward: 2997903.57
total_cost: 315385.00
total_trades: 78509
Sharpe: 0.785
-----------------------------------------
| time/                   |   

### 3.4 TD3 (Twin Delayed DDPG)

In [14]:
if if_using_td3:
    log_progress(3, 4, "RUNNING", "TD3 모델 학습 시작...")

    try:
        # GPU 메모리 확인
        print_gpu_memory("TD3 학습 시작 전")
        
        agent = DRLAgent(env=env_train)

        # TD3 하이퍼파라미터 (GPU 명시)
        TD3_PARAMS = {
            "batch_size": 100,
            "buffer_size": 1000000,
            "learning_rate": 0.001,
            "device": "cuda"
        }

        model_td3 = agent.get_model("td3", model_kwargs=TD3_PARAMS)

        # 로거 설정 (off-policy 알고리즘은 replay buffer를 사용하므로 CSV 로거 제외)
        tmp_path = RESULTS_DIR + '/td3'
        new_logger_td3 = configure(tmp_path, ["stdout", "tensorboard"])
        model_td3.set_logger(new_logger_td3)
        
        print(f"✅ TD3 모델이 GPU에서 학습됩니다: {model_td3.device}")
        print("📝 Custom Callback 사용 (rollout_buffer 에러 방지)")

        # 학습 실행 - Custom Callback 사용
        trained_td3 = model_td3.learn(
            total_timesteps=50000,
            tb_log_name='td3',
            callback=OffPolicyTensorboardCallback()
        )

        # 모델 저장
        save_path = TRAINED_MODEL_DIR + "/agent_td3"
        trained_td3.save(save_path)

        if os.path.exists(save_path + ".zip"):
            file_size = os.path.getsize(save_path + ".zip") / 1024
            print(f"\n✅ TD3 모델 저장 완료: {save_path}.zip ({file_size:.2f} KB)")
            
            # GPU 메모리 확인
            print_gpu_memory("TD3 학습 완료")
            
            log_progress(3, 4, "SUCCESS", f"TD3 학습 완료 및 저장 ({file_size:.2f}KB)")
        else:
            raise FileNotFoundError("모델 파일이 저장되지 않았습니다")

    except Exception as e:
        error_msg = f"TD3 학습 실패: {str(e)}"
        print(error_msg)
        log_progress(3, 4, "ERROR", error_msg)
else:
    print("TD3 학습 건너뜀")

[RUNNING] Phase 3.4: TD3 모델 학습 시작...

[GPU 메모리 - TD3 학습 시작 전]
  할당됨: 0.07 GB
  예약됨: 0.08 GB
  총 용량: 79.18 GB
  사용률: 0.1%

{'batch_size': 100, 'buffer_size': 1000000, 'learning_rate': 0.001, 'device': 'cuda'}
Using cuda device
Logging to results/td3
✅ TD3 모델이 GPU에서 학습됩니다: cuda
📝 Custom Callback 사용 (rollout_buffer 에러 방지)
day: 2892, episode: 90
begin_total_asset: 1000000.00
end_total_asset: 6384222.82
total_reward: 5384222.82
total_cost: 5090.50
total_trades: 36544
Sharpe: 0.972
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps             | 95        |
|    time_elapsed    | 121       |
|    total_timesteps | 11572     |
| train/             |           |
|    actor_loss      | 15.1      |
|    critic_loss     | 1.35e+03  |
|    learning_rate   | 0.001     |
|    n_updates       | 11471     |
|    reward          | 6.0478597 |
----------------------------------
----------------------------------
| time/              |       

### 3.5 SAC (Soft Actor-Critic)

In [12]:
if if_using_sac:
    log_progress(3, 5, "RUNNING", "SAC 모델 학습 시작...")

    try:
        # GPU 메모리 확인
        print_gpu_memory("SAC 학습 시작 전")
        
        agent = DRLAgent(env=env_train)

        # SAC 하이퍼파라미터 (GPU 명시)
        SAC_PARAMS = {
            "batch_size": 128,
            "buffer_size": 100000,
            "learning_rate": 0.0001,
            "learning_starts": 100,
            "ent_coef": "auto_0.1",
            "device": "cuda"
        }

        model_sac = agent.get_model("sac", model_kwargs=SAC_PARAMS)

        # 로거 설정 (off-policy 알고리즘은 replay buffer를 사용하므로 CSV 로거 제외)
        tmp_path = RESULTS_DIR + '/sac'
        new_logger_sac = configure(tmp_path, ["stdout", "tensorboard"])
        model_sac.set_logger(new_logger_sac)
        
        print(f"✅ SAC 모델이 GPU에서 학습됩니다: {model_sac.device}")
        print("📝 Custom Callback 사용 (rollout_buffer 에러 방지)")

        # 학습 실행 (70,000 timesteps) - Custom Callback 사용
        trained_sac = model_sac.learn(
            total_timesteps=70000,
            tb_log_name='sac',
            callback=OffPolicyTensorboardCallback()
        )

        # 모델 저장
        save_path = TRAINED_MODEL_DIR + "/agent_sac"
        trained_sac.save(save_path)

        if os.path.exists(save_path + ".zip"):
            file_size = os.path.getsize(save_path + ".zip") / 1024
            print(f"\n✅ SAC 모델 저장 완료: {save_path}.zip ({file_size:.2f} KB)")
            
            # GPU 메모리 확인
            print_gpu_memory("SAC 학습 완료")
            
            log_progress(3, 5, "SUCCESS", f"SAC 학습 완료 및 저장 ({file_size:.2f}KB)")
        else:
            raise FileNotFoundError("모델 파일이 저장되지 않았습니다")

    except Exception as e:
        error_msg = f"SAC 학습 실패: {str(e)}"
        print(error_msg)
        log_progress(3, 5, "ERROR", error_msg)
else:
    print("SAC 학습 건너뜀")

[RUNNING] Phase 3.5: SAC 모델 학습 시작...

[GPU 메모리 - SAC 학습 시작 전]
  할당됨: 0.00 GB
  예약됨: 0.00 GB
  총 용량: 79.18 GB
  사용률: 0.0%

{'batch_size': 128, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1', 'device': 'cuda'}
Using cuda device
Logging to results/sac
✅ SAC 모델이 GPU에서 학습됩니다: cuda
📝 Custom Callback 사용 (rollout_buffer 에러 방지)
---------------------------------
| time/              |          |
|    episodes        | 4        |
|    fps             | 70       |
|    time_elapsed    | 164      |
|    total_timesteps | 11572    |
| train/             |          |
|    actor_loss      | 1.42e+03 |
|    critic_loss     | 424      |
|    ent_coef        | 0.286    |
|    ent_coef_loss   | 137      |
|    learning_rate   | 0.0001   |
|    n_updates       | 11471    |
|    reward          | 8.673587 |
---------------------------------
----------------------------------
| time/              |           |
|    episodes        | 8         |
|    fps        

## Part 4. 학습 완료 및 검증

In [13]:
log_progress(4, 1, "RUNNING", "학습된 모델 검증 중...")

# 저장된 모델 파일 확인
print("\n=== 학습된 모델 파일 ===")
trained_models = []

for algo in ['a2c', 'ddpg', 'ppo', 'td3', 'sac']:
    model_path = f"{TRAINED_MODEL_DIR}/agent_{algo}.zip"
    if os.path.exists(model_path):
        file_size = os.path.getsize(model_path) / 1024  # KB
        print(f"✓ {algo.upper()}: {model_path} ({file_size:.2f} KB)")
        trained_models.append(algo.upper())
    else:
        print(f"✗ {algo.upper()}: 파일 없음")

log_progress(4, 1, "SUCCESS", f"모델 검증 완료 ({len(trained_models)}개 모델)")

[RUNNING] Phase 4.1: 학습된 모델 검증 중...

=== 학습된 모델 파일 ===
✓ A2C: trained_models/agent_a2c.zip (398.12 KB)
✓ DDPG: trained_models/agent_ddpg.zip (7640.21 KB)
✓ PPO: trained_models/agent_ppo.zip (582.98 KB)
✓ TD3: trained_models/agent_td3.zip (11470.41 KB)
✓ SAC: trained_models/agent_sac.zip (6386.46 KB)
[SUCCESS] Phase 4.1: 모델 검증 완료 (5개 모델)


## 완료!

DRL 모델 학습이 완료되었습니다.

**학습된 모델:**
- A2C, DDPG, PPO, TD3, SAC

**저장 위치:**
- `trained_models/agent_*.zip`

**다음 단계:**
`Stock_NeurIPS2018_3_Backtest_Fixed.ipynb`를 실행하여 백테스팅을 수행하세요.

In [14]:
# 최종 요약
log_progress(0, 0, "COMPLETED", "DRL 모델 학습 완료")

print("\n" + "="*60)
print("모델 학습 완료!")
print("="*60)
print(f"\n학습된 모델 ({len(trained_models)}개):")
for model in trained_models:
    print(f"  - {model}")
print(f"\n저장 위치: {TRAINED_MODEL_DIR}/")
print(f"로그 위치: {RESULTS_DIR}/")
print(f"\n다음 단계: Stock_NeurIPS2018_3_Backtest_Fixed.ipynb 실행")
print("="*60)

[COMPLETED] Phase 0.0: DRL 모델 학습 완료

모델 학습 완료!

학습된 모델 (5개):
  - A2C
  - DDPG
  - PPO
  - TD3
  - SAC

저장 위치: trained_models/
로그 위치: results/

다음 단계: Stock_NeurIPS2018_3_Backtest_Fixed.ipynb 실행
